# Ćwiczenie 2 — NumPy (Add‑on): Algorytmy sortowania (Insertion / Merge / Quick) + Python `sort` (Timsort)

Ten notebook jest **dodatkiem** do serii `02.* (NumPy)` z repozytorium *Python Data Science Handbook*.

## Cel (po tym ćwiczeniu)
Po ukończeniu notebooka student powinien:
1. Umieć opisać (intuicyjnie) i zaimplementować **Insertion Sort**, **Merge Sort** i **Quick Sort**.
2. Rozumieć różnicę: **stabilny vs niestabilny**, **in-place vs out-of-place**, oraz podstawowe złożoności czasowe/pamięciowe.
3. Wiedzieć, że wbudowany `list.sort()` / `sorted()` w Pythonie używa **Timsort** (hybryda: *merge sort + insertion sort*), i dlaczego to jest praktyczne.
4. Umieć użyć `%timeit` (i opcjonalnie `%prun` / `%lprun`) do porównania wydajności i znalezienia wąskich gardeł.

> Założenia środowiska: **Linux / WSL / Colab** (notebook Jupyter).


## Zasady pracy (styl RL)
- W miejscach oznaczonych `# TODO` uzupełnij kod.
- Po każdym większym zadaniu uruchom komórkę z testami (`assert`) i upewnij się, że widzisz `✅`.
- Staraj się pisać kod **czytelnie** (krótkie nazwy zmiennych tylko tam, gdzie to ma sens).


In [ ]:
# Setup
from __future__ import annotations

from dataclasses import dataclass
from collections import Counter
from typing import Callable, Iterable, List, Sequence, Tuple, TypeVar, Optional

import random
import math
import numpy as np

RANDOM_SEED = 0
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

T = TypeVar("T")

def is_sorted(seq: Sequence[T], key: Callable[[T], object] = lambda x: x) -> bool:
    """Return True if seq is non-decreasing w.r.t. key."""
    return all(key(seq[i]) <= key(seq[i+1]) for i in range(len(seq)-1))

def same_multiset(a: Sequence[T], b: Sequence[T]) -> bool:
    """Return True if a and b contain the same elements with the same multiplicities."""
    # Works for hashable elements. For unhashable, we won't use this helper.
    return Counter(a) == Counter(b)

print("Setup OK ✅")


## 0. Słowniczek: stabilność, in-place, złożoność

### Stabilność sortowania
Sortowanie jest **stabilne**, jeśli elementy o równych kluczach zachowują **oryginalną kolejność**.

Przykład (klucz = `age`):
- wejście: `[('A', age=20), ('B', age=20), ('C', age=30)]`
- stabilne sortowanie po `age` nie może zamienić miejscami `A` i `B`.

W Pythonie `sorted()` i `list.sort()` są **stabilne**.

### In-place vs out-of-place
- **in-place**: sortujemy “w miejscu” (modyfikujemy listę/array), zwykle O(1) dodatkowej pamięci (pomijając rekurencję).
- **out-of-place**: tworzymy nową listę wynikową (często prostsze), ale koszt pamięci jest większy.

### Złożoności (orientacyjnie)
| Algorytm | Średni czas | Najgorszy czas | Pamięć | Stabilny? |
|---|---:|---:|---:|---|
| Insertion sort | O(n²) | O(n²) | O(1) | ✅ |
| Merge sort | O(n log n) | O(n log n) | O(n) | ✅ |
| Quick sort | O(n log n) | O(n²) | O(log n) (rek.) | ❌ (zwykle) |

W kolejnych sekcjach zaimplementujesz te algorytmy i zobaczysz konsekwencje w praktyce.


## 1. Insertion Sort (sortowanie przez wstawianie)

### Intuicja
Budujemy posortowany prefiks listy:  
- krok `i=1`: wstawiamy element `a[1]` w odpowiednie miejsce w `[a[0]]`  
- krok `i=2`: wstawiamy element `a[2]` w odpowiednie miejsce w `[a[0], a[1]]`  
- itd.

### Pseudokod (klasyczny)
```text
for i = 1 .. n-1:
    x = a[i]
    j = i-1
    while j >= 0 and a[j] > x:
        a[j+1] = a[j]
        j = j - 1
    a[j+1] = x
```

### Trik implementacyjny (stabilność!)
Jeśli sortujesz po `key(x)`:
- aby algorytm był **stabilny**, w warunku przesuwania używaj porównania **`>`**, a nie `>=`.
  - `>`: elementy o równym kluczu *nie przeskakują* przez siebie ✅  
  - `>=`: elementy o równym kluczu mogą zmienić kolejność ❌

W zadaniu poniżej zaimplementujesz wersję, która działa jak `sorted(..., key=...)`.


In [ ]:
def insertion_sort(arr: Sequence[T], key: Callable[[T], object] = lambda x: x) -> List[T]:
    """Return a sorted copy of arr using insertion sort.

    Requirements:
    - MUST NOT modify the input sequence.
    - MUST be stable w.r.t. the provided key.
    - Return a Python list.

    Hint:
    - Use strict comparison `>` in the inner loop to keep stability.
    """
    a = list(arr)  # copy
    for i in range(1, len(a)):
        x = a[i]
        xk = key(x)
        j = i - 1
        # shift elements that are strictly larger (strict > keeps stability)
        while j >= 0 and key(a[j]) > xk:
            a[j + 1] = a[j]
            j -= 1
        a[j + 1] = x
    return a

# Quick self-check (no grading): insertion_sort([3,1,2]) should return [1,2,3]


In [ ]:
# Tests — Insertion sort (correctness + stability)
def _test_insertion_sort():
    # correctness on random ints
    for n in [0, 1, 2, 5, 20]:
        for _ in range(20):
            xs = [random.randint(-10, 10) for _ in range(n)]
            out = insertion_sort(xs)
            assert out == sorted(xs), f"Mismatch: {xs} -> {out}"
            assert xs == xs, "Input should not be modified (lists are copied)"  # sanity

    # stability test (tuples are hashable and easy to check)
    records = [("A", 2), ("B", 2), ("C", 1), ("D", 2), ("E", 1)]
    out = insertion_sort(records, key=lambda r: r[1])
    # expected order: all age=1 in original order C,E ; then all age=2 in original order A,B,D
    assert out == [("C", 1), ("E", 1), ("A", 2), ("B", 2), ("D", 2)], out

    assert is_sorted(out, key=lambda r: r[1])
    assert same_multiset(records, out)

_test_insertion_sort()
print("Insertion sort tests passed ✅")


## 2. Merge Sort (sortowanie przez scalanie)

### Intuicja
Dzielimy listę na połowy aż do rozmiaru 1, a potem **scalamy** posortowane fragmenty.

Kluczowe jest **scalanie (merge)** — jeśli zrobisz je dobrze, cały algorytm działa dobrze.

### Pseudokod (merge)
```text
merge(left, right):
    result = []
    i = 0; j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i]); i++
        else:
            result.append(right[j]); j++
    append leftovers from left and right
    return result
```

### Trik implementacyjny (stabilność w merge)
Aby mergesort był stabilny, przy równych kluczach wybieraj element z **lewej** listy.
W praktyce:
- warunek `if key(left[i]) <= key(right[j])` → wybierz `left[i]` ✅

Jeśli użyjesz `<` i przy równości weźmiesz z prawej, możesz złamać stabilność.


In [ ]:
def merge(left: Sequence[T], right: Sequence[T], key: Callable[[T], object] = lambda x: x) -> List[T]:
    """Merge two already-sorted sequences into a single sorted list (stable)."""
    out: List[T] = []
    i = 0
    j = 0

    while i < len(left) and j < len(right):
        # Stable: take from left when equal
        if key(left[i]) <= key(right[j]):
            out.append(left[i])
            i += 1
        else:
            out.append(right[j])
            j += 1

    # leftovers
    if i < len(left):
        out.extend(left[i:])
    if j < len(right):
        out.extend(right[j:])
    return out

def merge_sort(arr: Sequence[T], key: Callable[[T], object] = lambda x: x) -> List[T]:
    """Return a sorted copy of arr using merge sort (stable)."""
    a = list(arr)
    if len(a) <= 1:
        return a
    mid = len(a) // 2
    left = merge_sort(a[:mid], key=key)
    right = merge_sort(a[mid:], key=key)
    return merge(left, right, key=key)


In [ ]:
# Tests — Merge sort (correctness + stability)
def _test_merge_sort():
    for n in [0, 1, 2, 5, 50]:
        for _ in range(20):
            xs = [random.randint(-50, 50) for _ in range(n)]
            out = merge_sort(xs)
            assert out == sorted(xs), f"Mismatch: {xs} -> {out}"
            assert same_multiset(xs, out)

    records = [("A", 2), ("B", 2), ("C", 1), ("D", 2), ("E", 1)]
    out = merge_sort(records, key=lambda r: r[1])
    assert out == [("C", 1), ("E", 1), ("A", 2), ("B", 2), ("D", 2)], out

_test_merge_sort()
print("Merge sort tests passed ✅")


## 3. Quick Sort (sortowanie szybkie)

### Intuicja
Wybieramy *pivot* i dzielimy listę na dwie części:
- elementy mniejsze/równe pivotowi
- elementy większe od pivota

Następnie rekurencyjnie sortujemy obie części.

### Kluczowy element: **partition**
Najważniejszy “trik implementacyjny” w quicksort to **partycjonowanie in-place**:  
przechodzimy listę i przepychamy elementy na lewą/prawą stronę pivota.

### Pseudokod (Lomuto partition)
```text
partition(a, lo, hi):
    pivot = a[hi]
    i = lo
    for j = lo .. hi-1:
        if a[j] <= pivot:
            swap(a[i], a[j])
            i++
    swap(a[i], a[hi])
    return i   # final index of pivot
```

### Triki implementacyjne (ważne w praktyce)
1) **Wybór pivota**  
   - pivot jako ostatni element jest najprostszy, ale daje zły przypadek dla danych “prawie posortowanych”.  
   - praktycznie: losowy pivot albo “median-of-three” zmniejsza ryzyko O(n²).

2) **Ograniczanie głębokości rekurencji**  
   - sortuj najpierw mniejszą część rekurencyjnie, a większą “ogonowo” (tail recursion) → mniejsza szansa na `RecursionError`.

3) **Quicksort zwykle NIE jest stabilny**  
   - nawet jeśli sortuje poprawnie, elementy o równych kluczach mogą zmienić kolejność.


In [ ]:
def partition_lomuto(a: List[T], lo: int, hi: int, key: Callable[[T], object] = lambda x: x) -> int:
    """In-place Lomuto partition. Returns pivot final index.

    Requirements:
    - partition the slice a[lo:hi+1] around pivot=a[hi]
    - after partition:
        * for k in [lo, p-1] => key(a[k]) <= key(pivot)
        * for k in [p+1, hi] => key(a[k]) > key(pivot)
    """
    pivot_key = key(a[hi])
    i = lo
    for j in range(lo, hi):
        if key(a[j]) <= pivot_key:
            a[i], a[j] = a[j], a[i]
            i += 1
    a[i], a[hi] = a[hi], a[i]
    return i

def quicksort_inplace(a: List[T], lo: int = 0, hi: Optional[int] = None,
                     key: Callable[[T], object] = lambda x: x) -> None:
    """In-place quicksort (not necessarily stable).

    Requirements:
    - sort list a in-place.
    - use partition_lomuto as partitioning routine.
    - implement tail-recursion elimination to reduce recursion depth:
        recurse on smaller side first, then loop on the larger side.
    """
    if hi is None:
        hi = len(a) - 1
    # Tail-recursion elimination: loop over the larger side
    while lo < hi:
        p = partition_lomuto(a, lo, hi, key=key)
        left_size = p - lo
        right_size = hi - p
        # Recurse into the smaller partition first
        if left_size < right_size:
            # sort left part [lo, p-1]
            if lo < p - 1:
                quicksort_inplace(a, lo, p - 1, key=key)
            lo = p + 1  # continue with right part
        else:
            # sort right part [p+1, hi]
            if p + 1 < hi:
                quicksort_inplace(a, p + 1, hi, key=key)
            hi = p - 1  # continue with left part


In [ ]:
# Tests — Quick sort (correctness + partition invariants)
def _test_partition_invariants():
    xs = [3, 7, 2, 9, 1, 5]
    a = xs.copy()
    p = partition_lomuto(a, 0, len(a)-1)
    pivot = a[p]
    assert all(x <= pivot for x in a[:p]), (a, p)
    assert all(x > pivot for x in a[p+1:]), (a, p)
    assert same_multiset(xs, a)

def _test_quicksort_inplace():
    for n in [0, 1, 2, 10, 50]:
        for _ in range(20):
            xs = [random.randint(-100, 100) for _ in range(n)]
            a = xs.copy()
            quicksort_inplace(a)
            assert a == sorted(xs), f"Mismatch: {xs} -> {a}"

_test_partition_invariants()
_test_quicksort_inplace()
print("Quick sort tests passed ✅")


## 4. Co robi Python `sorted()` / `list.sort()`? (Timsort)

W CPython (standardowy Python), `sorted()` i `list.sort()` używają algorytmu **Timsort**.

### Krótko: czym jest Timsort?
To **stabilny** i **adaptacyjny** algorytm sortowania, który w dużym uproszczeniu łączy pomysły z:
- **merge sort** (scalanie posortowanych fragmentów),
- **insertion sort** (bardzo dobry na krótkich lub prawie posortowanych fragmentach).

### Schemtyczny opis (najważniejsze „triki”)
1) **Wykryj naturalne “runy”**  
   W realnych danych często są fragmenty już (prawie) posortowane. Timsort znajduje maksymalne rosnące odcinki (*runs*).

2) **Doprowadź krótkie runy do minimalnej długości (minrun)**  
   Krótkie runy są “dociągane” (dosortowane) przez insertion sort.

3) **Scalaj runy jak w merge sort**  
   Runy są trzymane na stosie i scalane tak, aby utrzymać rozsądne proporcje (inwarianty stosu).  
   Dzięki temu algorytm ma dobre zachowanie w praktyce i gwarancje w pesymistycznym przypadku.

> Poniżej zrobimy **toy wersję**: “natural merge sort” (bez pełnych inwariantów Timsort).  
> To ćwiczenie ma pokazać ideę: *runs + merge*.

### Pseudokod (uproszczony — idea)
```text
runs = find_non_decreasing_runs(a)

for each run:
    (optional) if run is short: insertion_sort(run)

while more than 1 run:
    merge runs pairwise into bigger runs

return merged run
```


In [ ]:
def find_non_decreasing_runs(arr: Sequence[T], key: Callable[[T], object] = lambda x: x) -> List[Tuple[int, int]]:
    """Return list of (start, end) indices for maximal non-decreasing runs in arr.

    Convention: run is arr[start:end] (end is exclusive).

    Example:
      arr = [1,2,2,0,5] -> runs: [(0,3), (3,5)]
    """
    n = len(arr)
    if n == 0:
        return []
    runs: List[Tuple[int, int]] = []
    start = 0
    for i in range(1, n):
        if key(arr[i-1]) <= key(arr[i]):
            continue
        # run ended at i
        runs.append((start, i))
        start = i
    runs.append((start, n))
    return runs

def natural_merge_sort(arr: Sequence[T], key: Callable[[T], object] = lambda x: x) -> List[T]:
    """Toy 'run-aware' stable sort: detect runs and merge them until one remains.

    This is NOT full timsort — it's just to practice the core idea.
    """
    a = list(arr)
    runs = find_non_decreasing_runs(a, key=key)

    # Convert runs to actual lists
    blocks: List[List[T]] = [a[s:e] for (s, e) in runs]

    # Optional: insertion-sort short runs to mimic minrun idea (kept tiny here)
    MINRUN = 8
    blocks = [insertion_sort(b, key=key) if len(b) < MINRUN else b for b in blocks]

    # Merge until one block remains
    while len(blocks) > 1:
        new_blocks: List[List[T]] = []
        for i in range(0, len(blocks), 2):
            if i + 1 < len(blocks):
                new_blocks.append(merge(blocks[i], blocks[i+1], key=key))
            else:
                new_blocks.append(blocks[i])
        blocks = new_blocks

    return blocks[0] if blocks else []


In [ ]:
# Tests — run detection + natural merge sort
def _test_find_runs():
    assert find_non_decreasing_runs([]) == []
    assert find_non_decreasing_runs([1]) == [(0,1)]
    assert find_non_decreasing_runs([1,2,2,0,5]) == [(0,3),(3,5)]
    assert find_non_decreasing_runs([3,2,1]) == [(0,1),(1,2),(2,3)]

def _test_natural_merge_sort():
    for n in [0, 1, 2, 10, 50]:
        for _ in range(20):
            xs = [random.randint(-20, 20) for _ in range(n)]
            out = natural_merge_sort(xs)
            assert out == sorted(xs)
    # stability check
    records = [("A", 2), ("B", 2), ("C", 1), ("D", 2), ("E", 1)]
    out = natural_merge_sort(records, key=lambda r: r[1])
    assert out == [("C", 1), ("E", 1), ("A", 2), ("B", 2), ("D", 2)]

_test_find_runs()
_test_natural_merge_sort()
print("Natural merge sort tests passed ✅")


## 5. Stabilność w praktyce: “multi‑key sort” bez pisania porównania

W Pythonie (i w wielu bibliotekach) sortowanie jest stabilne.  
To pozwala na bardzo prosty trik:

Aby posortować po dwóch kluczach:
1) sortujesz po **drugim** (mniej ważnym) kluczu,
2) potem stabilnie sortujesz po **pierwszym** (ważniejszym) kluczu.

Dzięki stabilności, elementy o równym pierwszym kluczu zachowają porządek po drugim kluczu.

Poniżej zaimplementujesz helper, który robi to przy użyciu naszego stabilnego `merge_sort`.


In [ ]:
def stable_two_key_sort(
    records: Sequence[T],
    primary_key: Callable[[T], object],
    secondary_key: Callable[[T], object],
) -> List[T]:
    """Stable two-key sort using only stable single-key sorting.

    Requirements:
    - MUST be stable w.r.t primary_key, then secondary_key.
    - Use merge_sort (stable) twice: first by secondary, then by primary.
    """
    tmp = merge_sort(records, key=secondary_key)
    out = merge_sort(tmp, key=primary_key)
    return out


In [ ]:
# Tests — stable two-key sort
def _test_stable_two_key_sort():
    records = [
        {"name": "A", "city": "WAW", "age": 30},
        {"name": "B", "city": "KRK", "age": 20},
        {"name": "C", "city": "WAW", "age": 20},
        {"name": "D", "city": "KRK", "age": 30},
        {"name": "E", "city": "WAW", "age": 20},
    ]
    out = stable_two_key_sort(
        records,
        primary_key=lambda r: r["city"],
        secondary_key=lambda r: r["age"],
    )
    # Compare to Python's stable sort with tuple key (city, age)
    expected = sorted(records, key=lambda r: (r["city"], r["age"]))
    assert out == expected

_test_stable_two_key_sort()
print("Two-key stable sort tests passed ✅")


## 6. (Opcjonalnie) Wydajność: porównaj algorytmy i zobacz *big‑O* w praktyce

Cel tej sekcji to **wyrobić nawyk**:
1) *najpierw poprawność* (testy ✅),  
2) dopiero potem *wydajność* (`%timeit` / `perf_counter`).

### Co zwykle zobaczysz
- Insertion sort szybko przegrywa dla większych n (O(n²)).
- Merge sort ma sensowną skalowalność (O(n log n)).
- Wbudowany `sorted()` jest zwykle najszybszy (Timsort + optymalizacje w C).

> Nie oczekujemy, że Twoje sorty będą “szybsze od Pythona”.  
> Tu chodzi o zrozumienie algorytmów i podstawowy workflow pomiaru.


In [ ]:
import time

def time_call(fn: Callable[..., object], *args, repeats: int = 5) -> float:
    """Return best-of-repeats wall time (seconds)."""
    best = float("inf")
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn(*args)
        dt = time.perf_counter() - t0
        best = min(best, dt)
    return best

# Data for benchmarking
def make_random_list(n: int, seed: int = 0) -> List[int]:
    rng = random.Random(seed)
    return [rng.randint(-10_000, 10_000) for _ in range(n)]

# TODO: fill these after you run the cell below (numbers in seconds)
T_INS_500 = None
T_MERGE_500 = None
T_BUILTIN_500 = None


In [ ]:
# Benchmark (keep n modest: insertion sort is O(n^2))
xs = make_random_list(500, seed=0)

# IMPORTANT: use copies for in-place quicksort
t_ins = time_call(insertion_sort, xs)
t_merge = time_call(merge_sort, xs)
t_builtin = time_call(sorted, xs)

print(f"insertion_sort: {t_ins:.6f}s")
print(f"merge_sort:     {t_merge:.6f}s")
print(f"sorted():       {t_builtin:.6f}s")

# TODO: assign the measured values to variables (copy from printed output)
T_INS_500 = t_ins
T_MERGE_500 = t_merge
T_BUILTIN_500 = t_builtin

# Optional (IPython): uncomment if you want
# %timeit insertion_sort(xs)
# %timeit merge_sort(xs)
# %timeit sorted(xs)


In [ ]:
# Tests — performance variables exist and make sense
assert isinstance(T_INS_500, float) and T_INS_500 > 0
assert isinstance(T_MERGE_500, float) and T_MERGE_500 > 0
assert isinstance(T_BUILTIN_500, float) and T_BUILTIN_500 > 0
print("Benchmark variables filled ✅")


## Podsumowanie / checklista oddania

✅ W tym notebooku powinno działać:
- `insertion_sort` (stabilny, z `key`)
- `merge` + `merge_sort` (stabilny, z `key`)
- `partition_lomuto` + `quicksort_inplace`
- `find_non_decreasing_runs` + `natural_merge_sort` (toy run-aware sort)
- `stable_two_key_sort`
- benchmark dla n=500 (wypełnione `T_*`)

### Do oddania
- notebook z przechodzącymi testami (`✅`)
- + krótki komentarz (3–6 zdań) w markdown:
  - które sorty są stabilne i dlaczego?
  - dlaczego quicksort może mieć zły przypadek?
  - co w praktyce daje “run-aware” podejście jak w Timsort?
